# CASMI v3: Contrastive Training + Retrieval
Fixed: only classes with >=5 spectra, 20K samples, fast training

In [ ]:
!pip install -q rdkit torch 2>&1 | tail -3


In [ ]:
import os, time, warnings, random, gc
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
class PeakEncoder(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_layers=4, max_peaks=200):
        super().__init__()
        self.peak_embed = nn.Linear(2, d_model)
        self.pos_embed = nn.Embedding(max_peaks, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True, dropout=0.1)
            for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, peaks):
        B, L, _ = peaks.shape
        x = self.peak_embed(peaks) + self.pos_embed(torch.arange(L, device=peaks.device)).unsqueeze(0)
        for l in self.layers: x = l(x)
        return self.norm(x).mean(dim=1)

class MolEncoder(nn.Module):
    def __init__(self, d_model=256, nbits=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nbits, d_model * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model)
        )
    def forward(self, fps):
        return self.net(fps)

class ContrastiveModel(nn.Module):
    def __init__(self, d_model=256, temp=0.07):
        super().__init__()
        self.spec_enc = PeakEncoder(d_model)
        self.mol_enc = MolEncoder(d_model)
        self.temp = temp
    def forward(self, peaks, fps):
        s = F.normalize(self.spec_enc(peaks), dim=1)
        m = F.normalize(self.mol_enc(fps), dim=1)
        logits = s @ m.T / self.temp
        labels = torch.arange(len(peaks), device=peaks.device)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        return loss, s, m


In [ ]:
from rdkit import DataStructs
from rdkit.Chem import AllChem

def batched_fps(smi_list, nbits=2048):
    result = np.zeros((len(smi_list), nbits), dtype=np.float32)
    for i, smi in enumerate(smi_list):
        if not smi: continue
        try:
            m = Chem.MolFromSmiles(smi)
            if m:
                fp = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=nbits)
                DataStructs.ConvertToNumpyArray(fp, result[i])
        except: pass
    return result

def augment(mzs, ints):
    m = mzs.copy(); v = ints.copy()
    m += np.random.normal(0, 0.005, len(m)).astype(np.float64)
    v *= (1 + np.random.normal(0, 0.05, len(v)).astype(np.float64))
    return m, np.clip(v, 0, None)

def process_peaks(mzs, ints, max_peaks=200):
    if len(mzs) == 0: return np.zeros((max_peaks, 2), dtype=np.float32)
    if len(mzs) > max_peaks:
        top = np.argsort(ints)[::-1][:max_peaks]
        mzs, ints = mzs[top], ints[top]
    order = np.argsort(mzs)
    mzs, ints = mzs[order], ints[order]
    if len(mzs) < max_peaks:
        pad = max_peaks - len(mzs)
        mzs = np.pad(mzs, (0, pad))
        ints = np.pad(ints, (0, pad))
    return np.stack([mzs, ints], 1).astype(np.float32)

class SpecDS(Dataset):
    def __init__(self, mzs, ints, fps, max_peaks=200):
        self.mzs=mzs; self.ints=ints; self.fps=fps; self.mp=max_peaks
    def __len__(self): return len(self.mzs)
    def __getitem__(self, i):
        m, v = augment(self.mzs[i], self.ints[i])
        peaks = process_peaks(m, v, self.mp)
        return torch.tensor(peaks), torch.tensor(self.fps[i], dtype=torch.float32)


In [ ]:
print('Loading data...')
DATA = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
train = pd.read_parquet(DATA+'/train.parquet')
print(f'  {len(train)} spectra')

smiles_raw = train['normalized_smiles'].tolist()
mzs_raw = train['ms2_mzs'].tolist()
ints_raw = train['ms2_normalized_intensities'].tolist()
k14_raw = train['inchikey14'].tolist()

# Keep ONLY molecules with >=5 spectra
smi_counts = defaultdict(list)
for i in range(len(smiles_raw)):
    if smiles_raw[i] and len(mzs_raw[i]) > 0:
        smi_counts[smiles_raw[i]].append(i)

multi = {s: idxs for s, idxs in smi_counts.items() if len(idxs) >= 5}
print(f'  {len(multi)} molecules with >=5 spectra')

# Take top 4000 by count, sample 5 per class
top_smi = sorted(multi.keys(), key=lambda s: len(multi[s]), reverse=True)[:4000]
train_idx = []
for s in top_smi:
    train_idx.extend(multi[s][:5])
print(f'  {len(top_smi)} classes x 5 samples = {len(train_idx)} total')
del train; gc.collect()


In [ ]:
print('Computing features...')
t0 = time.time()
fps = batched_fps([smiles_raw[i] for i in train_idx])
mzs = [np.array(mzs_raw[i], dtype=np.float64) for i in train_idx]
ints = [np.array(ints_raw[i], dtype=np.float64) for i in train_idx]
print(f'  Done in {time.time()-t0:.1f}s')

dataset = SpecDS(mzs, ints, fps)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
print(f'  {len(loader)} batches per epoch')


In [ ]:
print('Training...')
model = ContrastiveModel(d_model=256, temp=0.1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

best_loss = float('inf')
for epoch in range(30):
    model.train(); tl = 0; nb = 0; t0 = time.time()
    for peaks, fps_b in loader:
        peaks, fps_b = peaks.to(device), fps_b.to(device)
        loss, _, _ = model(peaks, fps_b)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item(); nb += 1
    sched.step()
    avg = tl / nb
    if avg < best_loss: best_loss = avg; torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
    print(f'  Epoch {epoch+1:2d}/30 | Loss: {avg:.4f} | {time.time()-t0:.0f}s')

print(f'Done. Best: {best_loss:.4f}')


In [ ]:
print('Building library embeddings...')
model.load_state_dict(torch.load('/kaggle/working/best_model.pt'))
model.eval()

seen_lib = set(); lib_smi = []; lib_k14 = []
for i in range(len(smiles_raw)):
    if smiles_raw[i] and smiles_raw[i] not in seen_lib:
        seen_lib.add(smiles_raw[i])
        lib_smi.append(smiles_raw[i])
        lib_k14.append(k14_raw[i])

lib_fps = batched_fps(lib_smi)
lib_loader = DataLoader(torch.tensor(lib_fps, dtype=torch.float32), batch_size=512, shuffle=False)
all_emb = []
with torch.no_grad():
    for fps_b in lib_loader:
        all_emb.append(F.normalize(model.mol_enc(fps_b.to(device)), dim=1).cpu().numpy())
lib_emb = np.concatenate(all_emb, axis=0)
print(f'  Library: {lib_emb.shape}')


In [ ]:
print('Loading test + retrieving...')
test = pd.read_parquet(DATA+'/test.parquet')
grouped = defaultdict(list); test_ids = []
for _, row in test.iterrows():
    mid = row['molecule_id']
    if mid not in grouped: test_ids.append(mid)
    grouped[mid].append((
        np.array(row['ms2_mzs'], dtype=np.float64),
        np.array(row['ms2_normalized_intensities'], dtype=np.float64)
    ))

results = []
for mid in test_ids:
    embs = []
    for m, v in grouped[mid]:
        peaks = process_peaks(m, v, max_peaks=200)
        t = torch.tensor(peaks, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = F.normalize(model.spec_enc(t), dim=1).cpu().numpy().reshape(-1)
            embs.append(emb)
    q = np.mean(embs, axis=0)
    norm_q = np.linalg.norm(q)
    if norm_q > 1e-6: q = q / norm_q
    sims = lib_emb @ q
    top = np.argsort(sims)[::-1][:100]
    seen = set(); cands = []
    for j in top:
        k = lib_k14[j]
        if k not in seen:
            seen.add(k)
            cands.append(lib_smi[j])
        if len(cands) >= 25: break
    results.append({'molecule_id': mid, 'smiles': ';'.join(cands)})

sub = pd.DataFrame(results)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission: {len(sub)} molecules')
print('=== DONE ===')
